<a href="https://colab.research.google.com/github/avanigargg/adk-samples/blob/main/push_agent_to_branch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CES Agent Export & SSM (Git) Push Pipeline
### Author: avanigarg@
### Team: GSD CES Incubation Team

This notebook automates the process of exporting a NextGen CES agent, preparing it for a CI/CD pipeline, and pushing it to a Secure Source Manager repository.

The pipeline performs the following steps:

*   **Export**: Exports a specified source agent from CES to a GCS bucket.
*   **Download & Unzip:** Downloads the exported .zip file from GCS to the local Colab environment.
*   **Parameterize:** Cleans the agent's JSON files, replacing the source agent's name (e.g., nga-agent-feature-x) with the canonical agent name (e.g., nga-agent).
*   **Clone:** Clones the target branch from Secure Source Manager repository and checks out the specified branch.
*   **Replace:** Deletes the old agent folder in the repo (if it exists) and copies the new, parameterized agent folder into its place.

## Run this cell once to install dependencies and authenticate.

In [ ]:
# Run this cell once to install dependencies and authenticate.
# For local execution, ensure libraries are installed and run `gcloud auth application-default login`.

import sys
import subprocess
# INSTALL_DEPS = False # Set to True if you need to install in Colab
INSTALL_DEPS = True # Set to True if you need to install in Colab
if INSTALL_DEPS:
    print("Installing dependencies...")
    # Add google-cloud-storage for GCS uploads/downloads
    subprocess.run([sys.executable, "-m", "pip", "install", "google-auth", "google-api-python-client", "requests", "GitPython", "google-cloud-storage", "--quiet"], check=True)
    print("Dependencies installed.")

    # Authenticate if in Colab
    try:
        from google.colab import auth
        print("Authenticating Colab user...")
        auth.authenticate_user()
        print("Colab user authenticated.")
    except ImportError:
        print("Not running in Colab or google.colab not found, skipping Colab auth. Ensure Application Default Credentials are set.")

print("Setup complete. Proceeding with script execution...")
# --- End of Setup ---

Installing dependencies...
Dependencies installed.
Authenticating Colab user...
Colab user authenticated.
Setup complete. Proceeding with script execution...


##Pipleine Code




### Import all necessary standard and third-party libraries

In [ ]:
import time
import zipfile
import os
import shutil
import git
import requests
import google.auth
from google.auth.transport.requests import Request
from google.api_core import exceptions
import re
import json
import base64
from urllib.parse import urlparse, urlunparse
import io
import datetime
from google.cloud import storage
import argparse

### Configurations


In [ ]:
# --- Configuration (Defaults & Command-line Arguments) ---

def parse_arguments():
    """Parses command-line arguments."""
    parser = argparse.ArgumentParser(description="Export a CES App to GCS, download, unzip, parameterize, rename, and push folder to a Git branch.")

    # Source GCP Args
    parser.add_argument("--source-project-id", required=True, help="GCP Project ID where the source app exists.")
    parser.add_argument("--source-location", required=True, help="Location of the source app (e.g., us).")
    parser.add_argument("--source-app-id", required=True, help="The App ID (UUID) of the source app to export.")

    # GCS Args
    parser.add_argument("--gcs-bucket", required=True, help="GCS bucket name for storing the export artifact (e.g., nga-cicd-artifacts).")
    # GCS object path will be constructed dynamically

    # Git Args
    parser.add_argument("--git-repo-url", required=True, help="HTTPS URL of the Google Cloud Source Repository.")
    parser.add_argument("--branch", required=True, help="The Git feature branch to push to (e.g., feature_avani).")
    parser.add_argument("--folder-in-repo", required=True, help="The desired *canonical* name of the agent folder in the Git repo (e.g., Movie_Expert_CICD).")
    parser.add_argument("--git-commit-message", default="CI: Automated export from feature agent", help="Commit message for Git.")
    parser.add_argument("--git-user-email", required=True, help="Git user email for the commit.")
    parser.add_argument("--git-user-name", required=True, help="Git user name for the commit.")


    args = parser.parse_args()

    # --- Construct dynamic paths (GCS path depends on app display name, fetched later) ---
    args.source_app_resource_name = f"projects/{args.source_project_id}/locations/{args.source_location}/apps/{args.source_app_id}"
    args.local_repo_path = "./temp_repo_clone_push" # Where Git repo is cloned
    # Local paths (download, unzip, rename) will be set dynamically in main() based on app name
    args.local_download_path = None # Set later
    args.local_unzip_path = None    # Set later (will be named after agent initially)
    args.renamed_unzip_path = None  # Set later (will be the target folder name)
    # args.gcs_artifact_uri will be set later in main() after fetching app name

    return args

### Define Helper Classes and functions

In [ ]:
# --- Helper Classes ---

class CESApp:
    """Handles CES REST API operations."""

    def __init__(self):
        try:
            self.creds, self.project = google.auth.default(
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            self.base_api_url = "https://ces.googleapis.com/v1beta/"
            self._refresh_token_if_needed()
            self.headers = {
                "Authorization": f"Bearer {self.access_token}",
                "Content-Type": "application/json",
            }
        except Exception as e:
            print(f"Error during CESApp initialization: {e}", file=sys.stderr)
            raise

    def _refresh_token_if_needed(self):
         if not self.creds or not self.creds.valid:
              print("Refreshing auth token...")
              auth_req = Request()
              self.creds.refresh(auth_req)
              if not self.creds or not self.creds.token:
                  raise Exception("Could not obtain access token.")
              self.access_token = self.creds.token
              self.headers = { # Update headers after refresh
                "Authorization": f"Bearer {self.access_token}",
                "Content-Type": "application/json",
              }
              print("Auth token refreshed.")

    def get_app(self, app_resource_name: str) -> dict:
        """Gets details of a specific app."""
        get_url = f"{self.base_api_url}{app_resource_name}"
        print(f"Getting app details for: {app_resource_name}...")
        self._refresh_token_if_needed()
        try:
            response = requests.get(get_url, headers=self.headers)
            response.raise_for_status()
            app_data = response.json()
            print("Successfully retrieved app details.")
            return app_data
        except Exception as e:
            print(f"Error calling getApp API: {e}", file=sys.stderr)
            raise

    def export_app_to_gcs(self, app_resource_name: str, gcs_uri: str) -> str:
        """
        Starts the app export to a GCS URI and returns the operation name.
        Uses JSON format.
        """
        export_url = f"{self.base_api_url}{app_resource_name}:exportApp"
        self._refresh_token_if_needed()

        # Payload based on ExportAppRequest documentation
        payload = {
            "export_format": "JSON",
            "gcs_uri": gcs_uri
        }
        print(f"Submitting export request for {app_resource_name} to {gcs_uri}...")

        try:
            response = requests.post(export_url, headers=self.headers, json=payload)
            response.raise_for_status()
            response_json = response.json()
            if "name" in response_json:
                return response_json["name"] # Operation name
            else:
                raise Exception(f"Export response did not contain 'name' (operation name). Response: {response_json}")
        except Exception as e:
            print(f"Error calling exportApp API: {e}", file=sys.stderr)
            raise


    def wait_for_operation(self, operation_name: str) -> dict:
        """Polls a long-running operation until done."""
        op_url = f"{self.base_api_url}{operation_name}"
        while True:
            try:
                self._refresh_token_if_needed()
                response = requests.get(op_url, headers=self.headers)
                response.raise_for_status()
                op_json = response.json()
                if op_json.get('done'):
                    print("Operation finished.")
                    if 'error' in op_json:
                        raise Exception(f"Operation failed: {op_json['error']}")
                    # Check for response, even if empty, indicates success without error
                    if 'response' in op_json:
                         print("Operation response received.")
                         return op_json
                    else:
                         # Handle cases where operation succeeded but response field is missing
                         print("Warning: Operation completed successfully but response field is missing. Assuming success based on 'done' status.")
                         return op_json # Still return the operation details
                else:
                    print("Operation in progress, waiting 10s...")
                    time.sleep(10)
            except Exception as e:
                print(f"Error polling operation: {e}", file=sys.stderr)
                raise

class Github:
    """Handles Git operations including cloning, commit, and push."""
    def __init__(self, access_token: str, user_email: str, user_name: str):
        self.access_token = access_token
        self.user_email = user_email
        self.user_name = user_name
        os.environ["GIT_TERMINAL_PROMPT"] = "0"

    def _get_authenticated_url(self, repo_url: str) -> str:
        parsed = urlparse(repo_url)
        netloc_with_token = f"oauth2:{self.access_token}@{parsed.netloc}"
        authed_url_parts = (parsed.scheme, netloc_with_token, parsed.path, parsed.params, parsed.query, parsed.fragment)
        return urlunparse(authed_url_parts)

    def clone_or_pull_repo(self, repo_url: str, local_repo_path: str, branch: str) -> git.Repo:
        """Clones repo if not exists, otherwise pulls the specified branch."""
        authed_repo_url = self._get_authenticated_url(repo_url)
        repo = None
        remote_branch_exists = False # Flag to track if remote branch exists

        if os.path.exists(local_repo_path):
            print(f"Repository already exists at {local_repo_path}. Fetching and checking out branch '{branch}'...")
            try:
                repo = git.Repo(local_repo_path)
                repo.remotes.origin.fetch()

                # Check if remote branch exists before trying to track/pull
                for ref in repo.remotes.origin.refs:
                    if ref.name == f'origin/{branch}':
                        remote_branch_exists = True
                        break

                # Ensure the branch exists locally and is tracking the remote if remote exists
                if branch in repo.heads:
                    repo.heads[branch].checkout()
                    if remote_branch_exists and repo.heads[branch].tracking_branch() is None:
                         print(f"Setting local branch '{branch}' to track 'origin/{branch}'...")
                         repo.heads[branch].set_tracking_branch(repo.remotes.origin.refs[branch])
                elif remote_branch_exists:
                    print(f"Creating local branch '{branch}' to track 'origin/{branch}'...")
                    repo.create_head(branch, repo.remotes.origin.refs[branch]).set_tracking_branch(repo.remotes.origin.refs[branch]).checkout()
                else:
                    print(f"Remote branch 'origin/{branch}' not found. Creating and checking out new local branch '{branch}'...")
                    repo.git.checkout('-b', branch)


                print(f"Checking out branch '{branch}'.")
                repo.git.checkout(branch) # Ensure checkout even if already exists

                # Only pull if the branch existed remotely
                if remote_branch_exists:
                    print(f"Pulling latest changes for branch '{branch}'...")
                    repo.remotes.origin.pull()
                    print("Pull successful.")
                else:
                    print(f"Branch '{branch}' is new locally, skipping pull.")

            except Exception as e:
                print(f"Error handling existing repository: {e}. Attempting clean clone.", file=sys.stderr)
                if os.path.exists(local_repo_path):
                     shutil.rmtree(local_repo_path)
                repo = None # Force re-clone

        if repo is None:
            print(f"Cloning branch '{branch}' from {repo_url} into {local_repo_path}...")
            try:
                repo = git.Repo.clone_from(authed_repo_url, local_repo_path, branch=branch)
                print(f"Successfully cloned branch '{branch}'.")
                remote_branch_exists = True # If clone worked, remote exists
            except git.GitCommandError as clone_err:
                 print(f"Warning: Cloning branch '{branch}' directly failed: {clone_err}. Trying fallback...")
                 try:
                      if os.path.exists(local_repo_path): # Clean up partial clone attempt
                           shutil.rmtree(local_repo_path)

                      print(f"Cloning default branch first...")
                      repo = git.Repo.clone_from(authed_repo_url, local_repo_path) # Clone default branch
                      print("Cloned default branch. Checking if target branch exists remotely...")
                      for ref in repo.remotes.origin.refs:
                           if ref.name == f'origin/{branch}':
                                remote_branch_exists = True
                                break
                      if remote_branch_exists:
                           print(f"Remote branch 'origin/{branch}' found. Creating local tracking branch...")
                           repo.create_head(branch, repo.remotes.origin.refs[branch]).set_tracking_branch(repo.remotes.origin.refs[branch]).checkout()
                           print(f"Successfully checked out existing remote branch '{branch}'.")
                      else:
                           print(f"Remote branch 'origin/{branch}' not found. Creating and checking out new local branch '{branch}'...")
                           repo.git.checkout('-b', branch)
                           print(f"Created and checked out new local branch '{branch}'. It will be pushed later.")
                 except Exception as fallback_err:
                      print(f"Error during fallback clone/branch creation: {fallback_err}", file=sys.stderr)
                      raise fallback_err

            except Exception as e:
                print(f"Error cloning repository: {e}", file=sys.stderr)
                raise
        return repo

    def commit_and_push(self, repo: git.Repo, commit_message: str, branch: str):
        """Adds all changes within the repo, commits, and pushes."""
        try:
            print("Configuring Git user...")
            repo.config_writer().set_value("user", "email", self.user_email).release()
            repo.config_writer().set_value("user", "name", self.user_name).release()

            print("Adding all changes...")
            repo.git.add(A=True)

            has_changes = False
            try:
                has_changes = repo.is_dirty(untracked_files=True)
            except Exception as check_err:
                 print(f"Warning: Error checking for git changes: {check_err}. Relying on index diff.")
                 has_changes = repo.index.diff("HEAD" if repo.head.is_valid() else None)

            if has_changes:
                print(f"Committing changes with message: '{commit_message}'")
                repo.index.commit(commit_message)
                print("Commit successful.")

                print(f"Pushing changes to origin/{branch}...")
                origin = repo.remote(name='origin')
                origin.push(refspec=f'{branch}:{branch}', set_upstream=True)
                print("Push successful.")
            else:
                print("No changes detected to commit.")

        except Exception as e:
            print(f"Error during commit/push: {e}", file=sys.stderr)
            if "non-fast-forward" in str(e):
                 print("Error: Non-fast-forward push detected...", file=sys.stderr)
            raise


class GcsUtils:
    """Handles GCS download operations."""
    def __init__(self):
        try:
            self.storage_client = storage.Client()
        except Exception as e:
            print(f"Error initializing GCS client: {e}", file=sys.stderr)
            raise

    def download_blob(self, bucket_name: str, source_blob_name: str, destination_file_name: str):
        """Downloads a blob from the bucket."""
        print(f"Downloading gs://{bucket_name}/{source_blob_name} to {destination_file_name}...")
        try:
            bucket = self.storage_client.bucket(bucket_name)
            blob = bucket.blob(source_blob_name)
            # Ensure destination directory exists before downloading
            os.makedirs(os.path.dirname(destination_file_name), exist_ok=True)
            blob.download_to_filename(destination_file_name)
            print("File downloaded successfully.")
        except exceptions.NotFound:
            print(f"Error: Blob {source_blob_name} not found in bucket {bucket_name}.", file=sys.stderr)
            raise
        except Exception as e:
            print(f"Error downloading from GCS: {e}", file=sys.stderr)
            raise

In [ ]:
# --- Helper Functions ---

def unzip_to_folder_name(zip_path: str, base_extract_path: str, expected_folder_name: str) -> str:
    """
    Extracts a zip file into the base_extract_path.
    It assumes the zip file *contains* the 'expected_folder_name' as its root.
    Returns the full path to the *extracted* folder.
    """
    # Define the path where the folder *will be* after extraction
    extract_to_path = os.path.join(base_extract_path, expected_folder_name)

    print(f"Preparing to extract in: {base_extract_path}...")
    # Clean up any old version of the folder *before* extracting
    if os.path.exists(extract_to_path):
        print(f"Removing existing extraction folder: {extract_to_path}...")
        shutil.rmtree(extract_to_path)

    # Ensure the base path exists (e.g., /content/)
    if not os.path.exists(base_extract_path):
        os.makedirs(base_extract_path)

    print(f"Extracting {zip_path} into {base_extract_path}...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            file_list = zip_ref.namelist()
            if not file_list:
                raise ValueError(f"Downloaded zip file is empty: {zip_path}")

            # Extract all contents into the base path
            zip_ref.extractall(base_extract_path)
            print(f"Extraction complete. Files extracted to {base_extract_path}.")

            # Verify the expected folder was created by the zip
            # Check for the folder name from the *file list* (case-sensitive)
            actual_extracted_folder = None
            if expected_folder_name in os.listdir(base_extract_path):
                 actual_extracted_folder = os.path.join(base_extract_path, expected_folder_name)

            # Fallback: check case-insensitively or for the first dir if exact match fails
            if not actual_extracted_folder:
                for item in os.listdir(base_extract_path):
                     item_path = os.path.join(base_extract_path, item)
                     if os.path.isdir(item_path) and item.lower() == expected_folder_name.lower():
                          print(f"Warning: Found folder '{item}' (case mismatch). Using this folder.")
                          actual_extracted_folder = item_path
                          break

            if not actual_extracted_folder:
                 # Check if maybe the zip contained a *different* top-level folder
                possible_folders = [item for item in os.listdir(base_extract_path) if os.path.isdir(os.path.join(base_extract_path, item))]
                if len(possible_folders) == 1:
                     actual_folder_name = possible_folders[0]
                     print(f"Warning: Expected folder '{expected_folder_name}' not found directly. Found '{actual_folder_name}' instead. Using this folder.")
                     actual_extracted_folder = os.path.join(base_extract_path, actual_folder_name)
                else:
                    # Check if files were extracted flat (no top folder)
                    if "app.json" in os.listdir(base_extract_path):
                         print(f"Warning: Zip file was flat. Renaming '{base_extract_path}' to '{extract_to_path}' is not supported. Check zip structure.")
                         # This scenario is problematic for this logic, but let's point to the base
                         return base_extract_path # This will likely fail downstream, but it's what was extracted
                    raise FileNotFoundError(f"Extraction finished, but the expected folder '{expected_folder_name}' was not found in {base_extract_path} and couldn't determine the correct extracted folder.")

            return actual_extracted_folder # Return the path to the folder created by the zip

    except zipfile.BadZipFile:
        print(f"Error: The downloaded file '{zip_path}' is not a valid zip file or is corrupted.", file=sys.stderr)
        raise
    except Exception as e:
        print(f"Error unzipping file: {e}", file=sys.stderr)
        raise

# --- NEW: Function to parameterize/clean files ---
def parameterize_agent_files(directory_path: str, find_string: str, replace_string: str):
    """
    Recursively finds all .json files in a directory and replaces all occurrences
    of 'find_string' with 'replace_string' inside them.
    """
    if not find_string:
         print(f"Skipping parameterization: find_string is empty.")
         return

    print(f"Parameterizing files: Replacing '{find_string}' with '{replace_string}' in all .json files...")
    file_count = 0
    replacement_count = 0

    for root, dirs, files in os.walk(directory_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                try:
                    # Read the file content
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()

                    # Perform replacement
                    if find_string in content:
                        original_content = content
                        content = content.replace(find_string, replace_string)

                        # Write the modified content back
                        with open(file_path, 'w', encoding='utf-8') as f:
                            f.write(content)

                        # Count replacements (approximate)
                        count = original_content.count(find_string)
                        replacement_count += count
                        file_count += 1
                        # print(f"  - Replaced {count} instance(s) in {file}")

                except Exception as e:
                    print(f"Warning: Could not parameterize file {file_path}: {e}", file=sys.stderr)

    print(f"Parameterization complete. Replaced {replacement_count} instances across {file_count} files.")
# --- End of new function ---


def replace_folder_in_repo(source_folder_path: str, repo_root_path: str, agent_folder_name_in_repo: str):
    """
    Removes the target agent folder in the repo (if exists) and copies
    the source folder into the repo root with the target name.
    """
    target_repo_folder_path = os.path.join(repo_root_path, agent_folder_name_in_repo)

    print(f"Preparing target folder in repo: {target_repo_folder_path}...")
    # Remove the target folder completely if it exists to ensure a clean slate
    if os.path.exists(target_repo_folder_path):
        print(f"Removing existing target folder in repo: {target_repo_folder_path}...")
        shutil.rmtree(target_repo_folder_path)

    print(f"Copying {source_folder_path} to {target_repo_folder_path}...")
    try:
        # Copy the entire source folder to the target path
        shutil.copytree(source_folder_path, target_repo_folder_path)
        print("Copying complete.")
    except Exception as e:
        print(f"Error copying agent contents: {e}", file=sys.stderr)
        raise

### This cell defines the main() function that orchestrates the entire pipeline from start to finish.

In [ ]:
# --- Main Execution ---

def main(args):
    """Runs the push-to-branch pipeline."""
    print("--- Starting Push Agent to Branch Pipeline ---")
    archive_path = None # Store path to downloaded zip
    repo = None         # Store Git repo object
    gcs_artifact_uri = None # To be constructed after getting app name
    safe_source_app_name = None # Store sanitized app name
    local_unzipped_source_path = None # Path to initially unzipped folder (e.g., /content/movie_expert...)
    renamed_local_path = None # Path to folder after renaming

    try:
        # 1. Instantiate clients
        print("Authenticating...")
        ces = CESApp()
        github = Github(
            access_token=ces.access_token,
            user_email=args.git_user_email,
            user_name=args.git_user_name
        )
        gcs = GcsUtils()
        print("Authentication successful.")

        # --- Get App details first ---
        print(f"\n--- Step 1a: Fetching source app display name ---")
        app_details = ces.get_app(args.source_app_resource_name)
        source_app_display_name = app_details.get("displayName")
        if not source_app_display_name:
            raise ValueError(f"Could not retrieve display name for app {args.source_app_resource_name}")
        print(f"Source app display name: '{source_app_display_name}'")

        # --- Construct GCS URI and Local Paths using the display name ---
        safe_source_app_name = re.sub(r'[^a-zA-Z0-9_-]', '_', source_app_display_name)
        if not safe_source_app_name: # Handle empty string case after sanitizing
             raise ValueError(f"Sanitized app name is empty for '{source_app_display_name}'")

        gcs_folder = "feature"
        gcs_object_name = f"{gcs_folder}/{safe_source_app_name}.zip"
        gcs_artifact_uri = f"gs://{args.gcs_bucket}/{gcs_object_name}"
        print(f"GCS Artifact URI determined: {gcs_artifact_uri}")

        # Define local paths dynamically
        content_dir = os.path.abspath("./content") # Standard writable directory in Colab
        os.makedirs(content_dir, exist_ok=True) # Ensure /content exists

        # Path where the zip file will be downloaded
        args.local_download_path = os.path.join(content_dir, f"{safe_source_app_name}.zip")
        # Base path for extraction (/content/)
        local_extract_base = content_dir
        # The folder name expected *inside* the zip, matching the source app name
        initial_unzip_folder_name = safe_source_app_name # e.g., movie_expert_cicd_feature_avani
        # The path where the folder will *actually* be after extraction
        local_unzipped_source_path = os.path.join(local_extract_base, initial_unzip_folder_name)
        # The path we *want* the folder to have before copying to Git
        renamed_local_path = os.path.join(local_extract_base, args.folder_in_repo) # e.g., /content/Movie_Expert_CICD

        print(f"Local download path: {args.local_download_path}")
        print(f"Base extraction path: {local_extract_base}")
        print(f"Expected extracted folder name: {initial_unzip_folder_name}")
        print(f"Path after extraction: {local_unzipped_source_path}")
        print(f"Path after rename: {renamed_local_path}")


        # 2. Start Export App to GCS
        print(f"\n--- Step 1b: Exporting app {args.source_app_resource_name} to GCS ---")
        operation_name = ces.export_app_to_gcs(
            app_resource_name=args.source_app_resource_name,
            gcs_uri=gcs_artifact_uri
        )
        print(f"Export operation started: {operation_name}")

        # 3. Wait for Export Operation
        final_export_operation = ces.wait_for_operation(operation_name)

        # Verify export success
        export_success = False
        if final_export_operation.get('done') and 'error' not in final_export_operation:
             response_data = final_export_operation.get('response', {})
             if response_data.get('appUri') == gcs_artifact_uri or \
                response_data.get('gcs_uri') == gcs_artifact_uri:
                  export_success = True
                  print(f"Export confirmed successful via response URI: {gcs_artifact_uri}")
             else:
                  print("Warning: Operation completed, but response field did not contain expected URI. Assuming success based on 'done' status.")
                  export_success = True

        if not export_success:
             raise Exception(f"Export operation finished but did not confirm success. Details: {final_export_operation}")


        # 4. Download Exported Zip from GCS
        print("\n--- Step 2: Downloading exported zip from GCS ---")
        gcs_uri_parts = urlparse(gcs_artifact_uri)
        gcs_bucket_name = gcs_uri_parts.netloc
        gcs_object_path = gcs_uri_parts.path.lstrip('/')
        archive_path = args.local_download_path # Use updated path

        # os.makedirs(os.path.dirname(archive_path), exist_ok=True) # Already created content_dir

        if os.path.exists(archive_path):
            os.remove(archive_path)
        gcs.download_blob(gcs_bucket_name, gcs_object_path, archive_path)

        # --- Step 5: Unzip locally to folder named after agent ---
        print("\n--- Step 3: Unzipping downloaded agent locally ---")
        # Extract into the base path (/content), expecting it to create the subfolder
        extracted_folder_path = unzip_to_folder_name(archive_path, local_extract_base, initial_unzip_folder_name)
        # Update local_unzipped_source_path just in case unzip found a different name
        local_unzipped_source_path = extracted_folder_path

        # --- NEW STEP: Parameterize (Clean) Files ---
        print(f"\n--- Step 4: Parameterizing agent files ---")
        parameterize_agent_files(
            directory_path=local_unzipped_source_path,
            find_string=safe_source_app_name, # e.g., "movie_expert_cicd_feature_avani"
            replace_string=args.folder_in_repo    # e.g., "movie_expert_cicd"
        )
        # --- End of new step ---


        if local_unzipped_source_path == renamed_local_path:
            print(f"Source folder path and target folder path are identical ('{renamed_local_path}'). Skipping rename.")
            # Ensure the path still exists (it shouldn't have been deleted)
            if not os.path.exists(renamed_local_path):
                 raise FileNotFoundError(f"Expected folder not found at {renamed_local_path} after unzip/parameterization.")
        else:
            # --- Original rename logic (now in 'else' block) ---
            if os.path.exists(renamed_local_path):
                 print(f"Removing existing folder at rename destination: {renamed_local_path}")
                 shutil.rmtree(renamed_local_path)

            # Ensure the source path from unzip exists before moving
            if not os.path.exists(local_unzipped_source_path):
                 raise FileNotFoundError(f"Folder to rename not found at {local_unzipped_source_path} after unzip.")

            print(f"Renaming '{local_unzipped_source_path}' to '{renamed_local_path}'...")
            shutil.move(local_unzipped_source_path, renamed_local_path)
            print("Rename complete.")

        # --- Step 7: Clone or Pull Git Repo/Branch ---
        print(f"\n--- Step 6: Cloning/Pulling Git branch '{args.branch}' ---")
        repo = github.clone_or_pull_repo(args.git_repo_url, args.local_repo_path, args.branch)

        # --- Step 8: Replace folder in Repo ---
        print("\n--- Step 7: Replacing agent folder in Git repository ---")
        replace_folder_in_repo(renamed_local_path, args.local_repo_path, args.folder_in_repo)

        # --- Step 9: Commit and Push Changes ---
        print("\n--- Step 8: Committing and pushing changes to Git ---")
        github.commit_and_push(repo, args.git_commit_message, args.branch)

        print("\n--- Pipeline Finished Successfully ---")

    except Exception as e:
        print(f"\n--- Pipeline Failed ---", file=sys.stderr)
        print(f"An error occurred: {e}", file=sys.stderr)
        sys.exit(1) # Exit with an error code
    finally:
         # Clean up local files
         print("\n--- Cleaning up ---")
         if os.path.exists(args.local_repo_path):
             shutil.rmtree(args.local_repo_path)
             print(f"Cleaned up {args.local_repo_path}.")
         # Use archive_path which might have been updated
         current_archive_path = getattr(args, 'local_download_path', None)
         if current_archive_path and os.path.exists(current_archive_path):
             os.remove(current_archive_path)
             print(f"Cleaned up {current_archive_path}.")
         # Clean up initial local unzip path if it still exists (e.g., if rename failed or didn't happen)
         current_unzip_path = getattr(args, 'local_unzip_path', None)
         if current_unzip_path and os.path.exists(current_unzip_path):
             shutil.rmtree(current_unzip_path)
             print(f"Cleaned up {current_unzip_path}.")
         # Clean up renamed local path
         current_renamed_path = getattr(args, 'renamed_unzip_path', None)
         if current_renamed_path and os.path.exists(current_renamed_path):
              shutil.rmtree(current_renamed_path)
              print(f"Cleaned up {current_renamed_path}.")
         print("Cleanup complete.")

##Update the arguments to run the pipleine.

In [ ]:
if __name__ == "__main__":
    # --- Check if running in Colab or parse args ---
    if 'google.colab' in sys.modules:
        print("Running in Colab, using manually passed arguments for testing.")
        # --- Manually passed Args for Colab Testing ---
        class Args:
            pass
        args = Args()
        args.source_project_id="planar-ember-381915" #@param
        args.source_location="us"
        args.source_app_id="e958928e-528d-4809-b9aa-09c801c24304" #@param # UPDATED App ID
        args.gcs_bucket="nga-cicd-artifacts"
        args.git_repo_url="https://nga-repo-26840227468-git.us-central1.sourcemanager.dev/ccai-platform-project/repo-nextgen-agent.git"
        args.branch="feature_avani" #@param # Branch to push TO
        args.folder_in_repo="demo_cicd_agent" #@param # TARGET folder name in repo
        args.git_commit_message="CI: Automated export from feature agent (Colab Test)"
        args.git_user_email="avanigarg@google.com" # Use your actual email for commit
        args.git_user_name="Avani Garg (via Colab)" # Use your actual name

        # Construct dynamic paths (GCS URI, local paths set later in main)
        args.source_app_resource_name = f"projects/{args.source_project_id}/locations/{args.source_location}/apps/{args.source_app_id}"
        args.local_repo_path = "./temp_repo_clone_push"
        # args.local_download_path, args.local_unzip_path, args.renamed_unzip_path set in main


        # Print args being used (GCS URI will be determined later)
        print("Using the following arguments (GCS URI & local paths determined after fetching app name):")
        # Print initial args, paths will be updated later
        temp_args_dict = vars(args).copy()
        # Remove paths that will be set dynamically
        temp_args_dict.pop('local_download_path', None)
        temp_args_dict.pop('local_unzip_path', None)
        temp_args_dict.pop('renamed_unzip_path', None)

        for k, v in temp_args_dict.items():
            if k != 'gcs_artifact_uri': # Don't print the URI yet
                print(f"  --{k.replace('_', '-')}= {v}")
        print("-" * 20)

        main(args)
    else:
        # If not in Colab, parse command-line arguments
        args = parse_arguments()
        main(args)

Running in Colab, using manually passed arguments for testing.
Using the following arguments (GCS URI & local paths determined after fetching app name):
  --source-project-id= planar-ember-381915
  --source-location= us
  --source-app-id= e958928e-528d-4809-b9aa-09c801c24304
  --gcs-bucket= nga-cicd-artifacts
  --git-repo-url= https://nga-repo-26840227468-git.us-central1.sourcemanager.dev/ccai-platform-project/repo-nextgen-agent.git
  --branch= feature_avani
  --folder-in-repo= demo_cicd_agent
  --git-commit-message= CI: Automated export from feature agent (Colab Test)
  --git-user-email= avanigarg@google.com
  --git-user-name= Avani Garg (via Colab)
  --source-app-resource-name= projects/planar-ember-381915/locations/us/apps/e958928e-528d-4809-b9aa-09c801c24304
  --local-repo-path= ./temp_repo_clone_push
--------------------
--- Starting Push Agent to Branch Pipeline ---
Authenticating...
Refreshing auth token...
Auth token refreshed.
Authentication successful.

--- Step 1a: Fetching 